# Paloma Humans Datasets

Create:
- Make European-only human HA-only pandemic swine dataset

Trees: Make 2 Fast Trees
- One with all pig sequences + human dataset
- One with only Paloma sequences + human dataset

## Housekeeping

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
from unidecode import unidecode
import importlib
import utils  
importlib.reload(utils)
from utils import * # If changing utils, must restart this file for changes to take effect

pd.options.mode.chained_assignment = None # suppress warnings when using slices to make new columns

In [2]:
# Directory paths

home = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/Paloma/"
downloads = home + "downloads/human_euro_seasonal_h1n1_01-01-1977--12-31-2008/" 
references = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu/references/"

complete_files = home + "complete_human/" 
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

# if not os.path.exists(complete_files + "deduplicated/"): # checking if the directory exists or not
#     os.makedirs(complete_files + "deduplicated/") # if the directory is not present then create it

os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

In [3]:
# Get metadata and sequences

gisaid_metadata = []
gisaid_fastas = []
for dirpath, dirs, files in os.walk(downloads):
    for file in files:
        file_name = os.path.join(dirpath, file) # .split("/")[-1]
        if ".fasta" in file_name:
            fasta = fasta_df(file_name, states_ref)
            # fasta_df["Accession"] = fasta_df["full_header"].apply(lambda x: x.split("|")[0].replace(">", ""))
            # # All sequences should be human
            # fasta_df["Isolate"] = fasta_df["full_header"].apply(lambda x: x.split("/")[2]) # if "human" in x else x.split("/")[3] if "/" in x else "unknown") #  if len(x.split("/")) > 2 else "unknown")
            # fasta_df["Subtype"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-5])
            # fasta_df["Geo_Location"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-4])
            # fasta_df["Collection_Date"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-3])
            # fasta_df["Host_Type"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-2])
            # fasta_df["Genotype"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-1])
            gisaid_fastas.append(fasta)
        else: # if ".xls" in file name
            metadata = pd.read_excel(file_name)
            gisaid_metadata.append(metadata)

# Concatenate metadata
metadata_concat = pd.DataFrame()
for metadata_file in gisaid_metadata:
    metadata_concat = pd.concat([metadata_concat, metadata_file])

# Concatenate fastas
fasta_concat = pd.DataFrame()
for fasta in gisaid_fastas:
    fasta_concat = pd.concat([fasta_concat, fasta])

print(metadata_concat)
print(fasta_concat)


            Isolate_Id PB2 Segment_Id PB1 Segment_Id PA Segment_Id  \
0     EPI_ISL_10656176            NaN            NaN           NaN   
1       EPI_ISL_357605            NaN            NaN           NaN   
2       EPI_ISL_357575            NaN            NaN           NaN   
3       EPI_ISL_356955            NaN            NaN           NaN   
4       EPI_ISL_356952            NaN            NaN           NaN   
...                ...            ...            ...           ...   
1093     EPI_ISL_20429            NaN            NaN           NaN   
1094     EPI_ISL_20428            NaN            NaN           NaN   
1095     EPI_ISL_20414            NaN            NaN           NaN   
1096     EPI_ISL_20412            NaN            NaN           NaN   
1097     EPI_ISL_20371            NaN            NaN           NaN   

             HA Segment_Id NP Segment_Id         NA Segment_Id  \
0        EPI1989388|05S502           NaN                   NaN   
1        EPI1447528|04S059 

## Make names

In [7]:
# Find "animals" (geographic locations indicating human sequence)

segment_fastas = []
unique_animals_all = []

unique_animals = sort_animals(fasta_concat) # Find unique animals
    # print("Animals: ", unique_animals)
unique_animals_all.append(unique_animals)

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Flatten unique_animals
every_unique_animal = []
for l in unique_animals_all:
    for animal in l:
        every_unique_animal.append(animal)

# Rename host type

unique_animals_set = list(set(every_unique_animal))
animals_df = pd.DataFrame(columns=["wild_avian", "domestic_avian", "cattle", "feline", "other_mammal", "human", "pet_food", "other"])
animals_df["other"] = unique_animals_set # to sort

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

print(common_animals)
print(len(common_animals))

different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

print(animals_ref)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    nan_row = pd.DataFrame([[np.nan] * len(animals_df.columns)], columns=animals_df.columns)
    for i in range(number_of_times_to_add_nan):
        animals_df = pd.concat([animals_df, nan_row], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

os.chdir(references)
animals_df.to_csv("animals_ref_to_sort.csv")

['bremen', 'stpetersburg', 'valladolid', 'toulouse', 'ulan-ude', 'poitiers', 'geneva', 'umea', 'rheinlandpfalz', 'sachsen', 'salamanca', 'baleares', 'chelyabinsk', 'groningen', 'khabarovsk', 'switzerland', 'st._petersburg', 'madrid', 'vladivostok', 'leningrad', 'parma', 'luxembourg', 'bulgaria', 'donetsk', 'hungary', 'berlin', 'voronezh', 'kaliningrad', 'thueringen', 'ukraine', 'leiden', 'perth', 'burgos', 'podgorica', 'lyon_gi', 'trieste', 'moscow', 'finland', 'cherkasy', 'prague', 'novi_sad', 'england', 'hamburg', 'nib', 'kursk', 'rheinland-pfalz', 'france', 'romania', 'perugia', 'austria', 'montenegro', 'olsztyn-pl', 'plzen', 'belgium', 'genoa', 'st_petersburg', 'paris', 'hannover', 'vilnus', 'bratislava', 'samara', 'st.petersburg', 'ostrava', 'navarra', 'montpellier', 'lisbon', 'ussr', 'sofia', 'greece', 'volos', 'czechoslovakia', 'grodno', 'netherlands', 'siena', 'norway', 'tula', 'arkhangelsk', 'zagreb', 'oslo', 'nordrhein-wesrfalen', 'ireland', 'denmark', 'bucuresti', 'thessalon

In [ ]:
fasta = fix_animals(fasta_concat, animals_df) # Fix animals first

# >EPI_ID|Isolate_name|subtype|collection_date|host_type

xls = metadata_concat.rename(columns={"Isolate_Id":"Identifier"})

# Merge metadata with fasta
fasta_meta = fasta_concat.merge(xls, how="right", on="Identifier")

print(fasta_meta)
# Those with missing metadata get dropped
fasta_meta = fasta_meta.dropna(subset=["Identifier", "Isolate_Name_x", "Subtype_x", "Geo_Location", "Date Collected", "Host_Type"])
# Those with embargos get dropped
fasta_meta = fasta_meta[fasta_meta["Publishing_Embargo_Until"].isna()]

# Rename sequences 
new_name = ">" + fasta_meta["Identifier"] + "|" + fasta_meta["Isolate_Name_x"] + "|" + fasta_meta["Subtype_x"] + "|" + fasta_meta["Geo_Location"] + "|" + fasta_meta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else x) + "|" + fasta_meta["Host_Type"] 
# print(fasta_seg["New_Name"])

fasta_meta["full_header"] = new_name

print(fasta_meta)


                                                 Header Isolate_Id  \
0     EPI_ISL_10656176|A/Perth/28/2005|A_/_H1N1|MP|2...         28   
1     EPI_ISL_10656176|A/Perth/28/2005|A_/_H1N1|HA|2...         28   
2     EPI_ISL_357605|A/Poitiers/2168/2003|A_/_H1N1|N...       2168   
3     EPI_ISL_357605|A/Poitiers/2168/2003|A_/_H1N1|H...       2168   
4     EPI_ISL_357575|A/La_Reunion/1394/2003|A_/_H1N1...       1394   
...                                                 ...        ...   
1939  EPI_ISL_20414|A/Moscow/27/2007|A_/_H1N1|HA|200...         27   
1940  EPI_ISL_20412|A/Moscow/37/2007|A_/_H1N1|NA|200...         37   
1941  EPI_ISL_20412|A/Moscow/37/2007|A_/_H1N1|HA|200...         37   
1942  EPI_ISL_20371|A/Cherkasy/22/2007|A_/_H1N1|NA|2...         22   
1943  EPI_ISL_20371|A/Cherkasy/22/2007|A_/_H1N1|HA|2...         22   

              Isolate_Name_x Subtype_x Segment Location_Header Geo_Location  \
0            A/Perth/28/2005      H1N1      MP           Perth        Perth   
1

## Create FASTA

In [10]:
fasta_meta = fasta_meta.rename(columns={"Sequence":"sequence"})
df_to_fasta(fasta_meta, "human_euro_seasonal_h1n1_01-01-1977--12-31-2008.fasta", complete_files)